# G02-A v5 — Original image-level thesis: HAM10000 / BCN20000 source evaluation

Kaggle GPU + Internet ON. Attach: **G01 completed output**, **J1 original protocol lock**, and **original thesis trained Keras output** (`last-part-of-the-previous-one` / complete `rotation_ssl_paperlike` weights). Optionally attach supervised five-seed weights. This notebook downloads the **exact original 224px Hugging Face images**, plus the official ISIC2019 ZIP *only to reconcile renamed images* when an audited mapping is not already available. Not independent source-external validation.

The uploaded G01 result has zero original-test joins, because original J1 IDs are Hugging Face filenames. This notebook does not accept an empty G01 image-aware manifest. It independently reconstructs the original J1 held-out group through image-content matching; ambiguous matches are excluded and recorded. It keeps the original Keras architectures and class output order. No training or ensemble selection.

**v4 archive correction:** Resolves repeated Hugging Face ZIP basenames through exact byte comparison, original J1 filepaths and archive metadata; saves a duplicate audit and refuses to guess between non-identical ambiguous images. The original J1 test IDs and labels remain unchanged.


## 1. Inputs, hashes, original J1 protocol, and duplicate-safe original-image download

In [1]:
import os,json,hashlib,zipfile,shutil,subprocess,time,io,gc
from pathlib import Path
import numpy as np,pandas as pd
from PIL import Image
from sklearn.metrics import balanced_accuracy_score,accuracy_score,f1_score
SEED=42;CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
INPUT=Path('/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data')
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
TEMP=Path('/kaggle/temp/g02a_sources' if Path('/kaggle/working').exists() else '/mnt/data/g02a_sources')
TEMP.mkdir(parents=True,exist_ok=True)
OUT=WORK/'g02a-image-level-ham-bcn-v2'
for sub in ['reports','predictions','tables','cache']:(OUT/sub).mkdir(parents=True,exist_ok=True)
START=time.monotonic();MAX_HOURS=8.5;CHUNK=384;EXTERNAL_BATCH_SIZE=24;IMG_SIZE=224;NUM_CLASSES=8
MODEL_NAMES=['SimCLR_NoTTA','SimCLR_SelectedTTA','Rotation','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
EXTERNAL_CACHE_DIR=OUT/'cache';os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
def req(ok,msg):
    if not bool(ok):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def writejson(p,x):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_suffix(p.suffix+'.tmp');t.write_text(json.dumps(x,indent=2,default=str,allow_nan=False));t.replace(p)
def one(filename,root=INPUT,optional=False):
    hits=sorted(p for p in root.rglob(filename) if p.is_file())
    if not hits and optional:return None
    req(len(hits)==1,f'Expected one {filename}; found {hits[:5]}')
    return hits[0]
def download(url,out,min_size):
    out=Path(out);out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and out.stat().st_size>=min_size:return out
    p=subprocess.run(['curl','-fL','--retry','8','--retry-delay','6','--connect-timeout','35','-C','-','-o',str(out),url],check=False)
    req(p.returncode==0 and out.is_file() and out.stat().st_size>=min_size,f'Download failed: {url} (curl={p.returncode})')
    return out
def zip_name(p):
    return str(p).replace(chr(92),'/').lstrip('./').strip()
def original_filepaths():
    paths={}
    candidates=sorted(INPUT.rglob('original_80_10_10_locked_split_with_provenance.csv'))
    if len(candidates)==1:
        a=pd.read_csv(candidates[0],dtype=str,keep_default_na=False)
        req({'image_id','original_filepath','split'}.issubset(a.columns),'Wrong J1 provenance CSV')
        a=a.loc[a['split'].str.lower().str.strip().eq('test')]
        paths.update(dict(zip(a.image_id,a.original_filepath)))
    elif len(candidates)>1:raise RuntimeError(f'Ambiguous J1 provenance CSVs: {candidates}')
    if not paths:
        js=sorted(INPUT.rglob('rotation_fixed_train_val_test_split.json'))
        js=[p for p in js if 'rotation_ssl_paperlike' in str(p)] or js
        if len(js)==1:
            data=json.loads(js[0].read_text())
            paths.update({Path(str(v['filepath'])).stem:str(v['filepath']) for v in data['test']})
        elif len(js)>1:raise RuntimeError(f'Ambiguous original split JSONs: {js}')
    if paths:
        req(set(original_test.image_id).issubset(paths),'Original J1 split differs from attached source provenance')
    print('Original path provenance:',len(paths),'test entries')
    return paths
def archive_metadata(z):
    lookup={}
    for info in z.infolist():
        if Path(info.filename).name!='metadata.jsonl' or info.is_dir():continue
        for raw in z.read(info).decode('utf-8-sig').splitlines():
            if not raw.strip():continue
            r=json.loads(raw)
            fn=next((r[k] for k in ['file_name','filename','path','filepath','image'] if k in r),None)
            if not fn:continue
            key=Path(zip_name(fn)).stem
            lookup.setdefault(key,[]).append((zip_name(fn),str(r.get('label',''))))
    return lookup
def suffix_score(old,member):
    a=zip_name(old).split('/');b=zip_name(member).split('/')
    n=0
    for x,y in zip(reversed(a),reversed(b)):
        if x.lower()!=y.lower():break
        n+=1
    return n
def members_of(archive,expected_ids=None,original_paths=None,expected_labels=None,audit_name='archive'):
    expected=set(expected_ids) if expected_ids is not None else None
    valid_ext={'.jpg','.jpeg','.png','.webp','.bmp'}
    with zipfile.ZipFile(archive) as z:
        groups={}
        for info in z.infolist():
            if info.is_dir() or Path(info.filename).suffix.lower() not in valid_ext:continue
            key=Path(info.filename).stem
            if expected is None or key in expected:groups.setdefault(key,[]).append(info)
        if expected is not None:
            missing=sorted(expected-set(groups))
            req(not missing,f'{audit_name}: missing {len(missing)} image IDs in ZIP, examples={missing[:10]}')
        meta=archive_metadata(z)
        paths=original_paths or {};labels=expected_labels or {}
        mapping={};audit=[];unresolved=[]
        for key,items in sorted(groups.items()):
            selected=None;reason='unique'
            # Multiple ZIP directory entries with the same *complete path*: extraction keeps the last one.
            unique={zip_name(x.filename):x for x in items}
            entries=list(unique.values())
            if len(entries)==1:
                selected=entries[0];reason='unique_path' if len(items)==1 else 'same_path_last_entry'
            else:
                digests={x.filename:hashlib.sha256(z.read(x)).hexdigest() for x in entries}
                if len(set(digests.values()))==1:
                    selected=sorted(entries,key=lambda x:x.filename)[0];reason='byte_identical'
                if selected is None and key in paths:
                    scores=[suffix_score(paths[key],x.filename) for x in entries]
                    top=max(scores)
                    if top>=2 and scores.count(top)==1:
                        selected=entries[scores.index(top)];reason='original_j1_filepath'
                if selected is None and key in meta:
                    choices=[]
                    for x in entries:
                        for fn,label in meta[key]:
                            if key in labels and label!=str(labels[key]):continue
                            s=suffix_score(fn,x.filename)
                            if s>=2:choices.append((s,x))
                    if choices:
                        best=max(t for t,_ in choices)
                        finalists={x.filename:x for t,x in choices if t==best}
                        if len(finalists)==1:
                            selected=next(iter(finalists.values()));reason='metadata_path_label'
            if len(items)>1:
                for x in items:
                    audit.append({'stem':key,'member':x.filename,'file_size':x.file_size,'crc32':x.CRC,
                                  'selected_member':selected.filename if selected else '',
                                  'resolution':reason if selected else 'UNRESOLVED',
                                  'original_filepath':paths.get(key,''),'expected_label':str(labels.get(key,''))})
            if selected is None:unresolved.append(key)
            else:mapping[key]=selected
        report=OUT/'reports'/f'{audit_name}_duplicate_members.csv'
        if audit:pd.DataFrame(audit).to_csv(report,index=False)
        print(f'{audit_name}: requested={len(groups)}, repeated_stems={len(set(r["stem"] for r in audit))}, '
              f'unresolved={len(unresolved)}; report={report if audit else "none"}')
        req(not unresolved,f'{audit_name}: ambiguous non-identical images for {len(unresolved)} IDs; '
            f'examples={unresolved[:12]}. Inspect {report}; no arbitrary ZIP entry selected. '
            'Attach original J1 provenance CSV or rotation split JSON if needed.')
        return mapping
marker=one('g01_status.json');G01=marker.parent.parent
info=json.loads(marker.read_text())
req(info.get('status')=='COMPLETED' and info.get('protocol_ready'),'Attach completed G01 output')
cohort=pd.read_csv(G01/'manifests/all_sources_with_splits.csv',dtype=str,keep_default_na=False)
req({'image_id','label','source','source_group'}.issubset(cohort.columns),'Unexpected G01 metadata')
req(cohort.image_id.is_unique and cohort.source.isin(['HAM10000','BCN20000']).all(),'Malformed G01 source membership')
# J1 IDs use the original 224px Hugging Face filenames: they may not be ISIC IDs.
j1=one('original_80_10_10_locked_ids.csv')
original=pd.read_csv(j1,dtype=str,keep_default_na=False)
req({'image_id','split','label','class_name'}.issubset(original.columns),'Wrong original J1 split CSV')
original['split']=original['split'].str.lower().str.strip().replace({'val':'validation'})
original['label']=pd.to_numeric(original['label'],errors='raise').astype(int)
original['class_name']=original['class_name'].str.strip().str.upper()
req(original.image_id.is_unique and len(original)==25331,'J1 split identity/count invalid')
req(original.split.value_counts().to_dict()=={'train':20264,'validation':2533,'test':2534},'J1 counts differ from original thesis')
req(original.label.between(0,7).all() and (original.class_name==original.label.map(dict(enumerate(CLASSES)))).all(),'J1 class order mismatch')
original_test=original.loc[original.split.eq('test')].copy().reset_index(drop=True)
print('G01 original ISIC-ID join test rows:',int(cohort.get('image_test',pd.Series(dtype=str)).astype(str).str.lower().eq('true').sum()))
print('J1 original test:',len(original_test),'IDs:',original_test.image_id.head(4).tolist())
HF_URL='https://huggingface.co/datasets/MKZuziak/ISIC_2019_224/resolve/main/data.zip'
hf_candidates=[p for p in INPUT.rglob('data.zip') if 'isic' in str(p).lower() and p.is_file()]
HF_ARCHIVE=sorted(hf_candidates)[0] if len(hf_candidates)==1 else download(HF_URL,TEMP/'hf_isic_2019_224_data.zip',90*10**6)
J1_PATHS=original_filepaths()
HF_MEMBERS=members_of(HF_ARCHIVE,expected_ids=original_test.image_id.tolist(),
    original_paths=J1_PATHS,expected_labels=dict(zip(original_test.image_id,original_test.label)),
    audit_name='huggingface')
req(set(original_test.image_id).issubset(HF_MEMBERS),'The original J1 test IDs are not in Hugging Face ISIC_2019_224 data.zip; attach the correct source artifact')
print('HF archive:',HF_ARCHIVE,'J1 test images resolved:',len(HF_MEMBERS))

G01 original ISIC-ID join test rows: 0
J1 original test: 2534 IDs: ['ISIC_Resize_15566', 'ISIC_Resize_754', 'ISIC_Resize_4162', 'ISIC_Resize_21881']
Original path provenance: 2534 test entries
huggingface: requested=2534, repeated_stems=0, unresolved=0; report=none
HF archive: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/isic2019_rotation_local/raw_download/data.zip J1 test images resolved: 2534


## 2. Audited image-content crosswalk: original J1 files → HAM/BCN ISIC IDs

In [2]:
# Verify mapping using content, not the ordering of metadata or assumptions about archive names.
# A matching ISIC ID is accepted directly. Other J1 IDs require an image-fingerprint match.
import cv2
from PIL import ImageOps
POPCNT=np.unpackbits(np.arange(256,dtype=np.uint8)[:,None],axis=1).sum(1).astype(np.uint8)
def fingerprint(buf):
    with Image.open(io.BytesIO(buf)) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        a=np.asarray(im.resize((32,32),Image.Resampling.BILINEAR),dtype=np.uint8)
        g=cv2.cvtColor(a,cv2.COLOR_RGB2GRAY).astype('float32')
        d=cv2.dct(g)[:8,:8];median=np.median(d.reshape(-1)[1:])
        p=np.packbits((d>median).reshape(-1).astype('uint8'),bitorder='big').view('>u8')[0].astype('uint64')
        arr=np.asarray(im.resize((9,8),Image.Resampling.BILINEAR).convert('L'),dtype='uint8')
        h=np.packbits((arr[:,:-1]>arr[:,1:]).reshape(-1).astype('uint8'),bitorder='big').view('>u8')[0].astype('uint64')
        rgb=np.asarray(im.resize((16,16),Image.Resampling.BILINEAR),dtype='uint8')
        return p,h,rgb

def fingerprint_archive(archive,keys,members):
    ph=np.empty(len(keys),dtype='uint64');dh=np.empty(len(keys),dtype='uint64')
    rgb=np.empty((len(keys),16,16,3),dtype='uint8')
    with zipfile.ZipFile(archive) as z:
        for k,key in enumerate(keys):
            ph[k],dh[k],rgb[k]=fingerprint(z.read(members[key]))
            if (k+1)%2500==0:print('Hashed',k+1,'/',len(keys))
    return ph,dh,rgb

def hamming64(x,arr):
    b=np.bitwise_xor(np.uint64(x),arr).view('uint8').reshape(-1,8)
    return POPCNT[b].sum(1).astype(np.int16)

def content_match(query,reference,ref_ids,ref_source,ref_labels,hf_ids,hf_labels):
    hp,hd,hrgb=query;rp,rd,rrgb=reference
    rows=[]
    for k,id_ in enumerate(hf_ids):
        d1=hamming64(hp[k],rp);d2=hamming64(hd[k],rd)
        raw=d1*2+d2
        selected=np.argsort(raw,kind='stable')[:12]
        candidates=[]
        for ix in selected:
            mad=float(np.abs(hrgb[k].astype('int16')-rrgb[ix].astype('int16')).mean())
            candidates.append((float(raw[ix])+mad*.15,mad,int(d1[ix]),int(d2[ix]),int(ix)))
        candidates.sort()
        top=candidates[0];next_=candidates[1]
        top_score,mad,pa,da,ix=top
        valid=pa<=12 and da<=18 and mad<=28
        ambiguous=valid and any(z[2]<=12 and z[3]<=18 and z[1]<=28 and z[4]!=ix and z[0]-top_score<=3.0 for z in candidates[1:])
        status='AMBIGUOUS' if ambiguous else ('MATCHED' if valid else 'UNMATCHED')
        if status=='MATCHED' and str(hf_labels[k])!=str(ref_labels[ix]):status='LABEL_CONFLICT'
        rows.append({'hf_id':id_,'image_id':ref_ids[ix] if valid else '',
            'source':ref_source[ix] if valid else 'OTHER','status':status,
            'phash_hamming':pa,'dhash_hamming':da,'thumbnail_mad':round(mad,4),
            'top_score':round(top_score,3),'second_score':round(next_[0],3),
            'j1_label':str(hf_labels[k]),'isic2019_label':str(ref_labels[ix]) if valid else ''})
    return pd.DataFrame(rows)

crosswalk_path=OUT/'reports/g02a_hf_to_isic_crosswalk.csv'
prior=sorted(INPUT.rglob('g02a_hf_to_isic_crosswalk.csv'))
if prior:
    req(len(prior)==1,'Ambiguous previous crosswalk')
    cross=pd.read_csv(prior[0],dtype=str,keep_default_na=False)
    req(set(cross.hf_id)==set(original_test.image_id),'Old crosswalk belongs to another J1 split')
    cross.to_csv(crosswalk_path,index=False)
    print('Reused previous crosswalk',prior[0])
elif crosswalk_path.exists():
    cross=pd.read_csv(crosswalk_path,dtype=str,keep_default_na=False)
    req(set(cross.hf_id)==set(original_test.image_id),'Stale crosswalk belongs to another J1 split')
else:
    official_url='https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip'
    official_candidates=sorted(p for p in INPUT.rglob('ISIC_2019_Training_Input.zip') if p.is_file())
    official=official_candidates[0] if len(official_candidates)==1 else download(official_url,TEMP/'ISIC_2019_Training_Input.zip',5*10**9)
    official_members=members_of(official,expected_ids=cohort.image_id.tolist(),audit_name='isic2019')
    req(set(cohort.image_id).issubset(official_members),'G01 official image IDs absent from official ISIC 2019 archive')
    ids=cohort.image_id.tolist();reference_cache=OUT/'cache/official_ham_bcn_fingerprints.npz'
    if reference_cache.exists():
        with np.load(reference_cache,allow_pickle=False) as z:
            req(np.array_equal(z['image_id'].astype(str),np.asarray(ids)),'Stale fingerprint references')
            rp,rd,rrgb=z['phash'],z['dhash'],z['thumbnail']
    else:
        rp,rd,rrgb=fingerprint_archive(official,ids,official_members)
        np.savez_compressed(reference_cache,image_id=np.asarray(ids),phash=rp,dhash=rd,thumbnail=rrgb)
    hf_ids=original_test.image_id.tolist()
    hp,hd,hrgb=fingerprint_archive(HF_ARCHIVE,hf_ids,HF_MEMBERS)
    cross=content_match((hp,hd,hrgb),(rp,rd,rrgb),ids,cohort.source.tolist(),cohort.label.tolist(),hf_ids,original_test.class_name.tolist())
    cross.to_csv(crosswalk_path,index=False)
# Preserve all crosswalk decisions, but never score images with uncertain or conflicting identities.
req(cross.hf_id.is_unique and len(cross)==len(original_test),'Invalid J1 test crosswalk')
req({'hf_id','image_id','status'}.issubset(cross.columns),'Invalid crosswalk columns')
req(set(cross.hf_id)==set(original_test.image_id),'Crosswalk does not match locked J1 test')
ref_labels=cohort.set_index('image_id')['label'].to_dict()
j1_labels=original_test.set_index('image_id')['class_name'].to_dict()
accepted=cross.status.eq('MATCHED')
unknown=accepted&~cross.image_id.isin(ref_labels)
if unknown.any():cross.loc[unknown,'status']='UNKNOWN_OFFICIAL_ID'
accepted=cross.status.eq('MATCHED')
conflict=accepted&cross.hf_id.map(j1_labels).ne(cross.image_id.map(ref_labels))
if conflict.any():cross.loc[conflict,'status']='LABEL_CONFLICT'
accepted=cross.status.eq('MATCHED')
duplicated=accepted&cross.image_id.duplicated(keep=False)
if duplicated.any():
    dup_ids=set(cross.loc[duplicated,'image_id'])
    cross.loc[accepted&cross.image_id.isin(dup_ids),'status']='DUPLICATE_OFFICIAL_ID'
# A prior incorrect run may have saved a crosswalk. Always re-audit and overwrite it.
cross.to_csv(crosswalk_path,index=False)
print('G02-A crosswalk audited:',cross.status.value_counts().to_dict())
issues=cross.loc[~cross.status.eq('MATCHED')].copy()
issues.to_csv(OUT/'reports/g02a_unresolved_identity_records.csv',index=False)
print('Excluded unresolved/conflicting image identities:',len(issues),'of',len(cross))
# Excluding unresolved associations is not a label correction. No labels are changed.
match=cross.loc[cross.status.eq('MATCHED'),['hf_id','image_id']].copy()
req(match.hf_id.is_unique and match.image_id.is_unique,'Accepted mapping is not one-to-one')
req(not match.empty,'No uniquely mapped original test images; cannot perform source evaluation')
df=match.merge(cohort.drop(columns=['image_split','image_test'],errors='ignore'),on='image_id',validate='one_to_one')
df=df.merge(original_test[['image_id','label','class_name']].rename(
    columns={'image_id':'hf_id','label':'j1_label','class_name':'j1_class'}),on='hf_id',validate='one_to_one')
req((df.label==df.j1_class).all(),'Accepted mappings contain diagnostic disagreement')
req(df.source.isin(['HAM10000','BCN20000']).all(),'Invalid source assignment')
df['label_id']=df.label.map({c:i for i,c in enumerate(CLASSES)}).astype(int)
df=df.sort_values(['source','image_id']).reset_index(drop=True)
source_counts=df.source.value_counts().to_dict()
req(all(source_counts.get(c,0)>0 for c in ['HAM10000','BCN20000']),
    f'One source has no identifiable test images: {source_counts}')
coverage=pd.DataFrame({'J1_test_images':[len(original_test)],'accepted':[len(df)],
    'unresolved':[len(issues)],'coverage_fraction':[len(df)/len(original_test)]})
coverage.to_csv(OUT/'reports/g02a_crosswalk_coverage.csv',index=False)
df.to_csv(OUT/'reports/g02a_verified_test_cohort.csv',index=False)
writejson(OUT/'reports/g02a_mapping_provenance.json',{
    'j1_split_sha256':sha(j1),'g01_cohort_sha256':info['cohort_sha256'],
    'hf_archive':str(HF_ARCHIVE),'n_original_test':len(original_test),
    'n_ham_bcn_matched':len(df),'matched_by_source':source_counts,
    'excluded_by_reason':issues.status.value_counts().to_dict(),
    'coverage_fraction':len(df)/len(original_test),
    'method':'Conservative approximate image-fingerprint matching, one-to-one and label consistency; unresolved cases excluded',
    'limitation':'Source-specific sample is a verified-label identifiable subset, not the entire original thesis test set. Perceptual matches remain approximate.'})
print('G02-A accepted cohort:',len(df),source_counts,'test coverage:',f'{len(df)/len(original_test):.2%}')
print('G02-A per-class:',df.groupby(['source','label']).size().unstack(fill_value=0).to_string())


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9318M  100 9318M    0     0  59.7M      0  0:02:36  0:02:36 --:--:-- 65.0M


isic2019: requested=22428, repeated_stems=0, unresolved=0; report=none
Hashed 2500 / 22428
Hashed 5000 / 22428
Hashed 7500 / 22428
Hashed 10000 / 22428
Hashed 12500 / 22428
Hashed 15000 / 22428
Hashed 17500 / 22428
Hashed 20000 / 22428
Hashed 2500 / 2534
G02-A crosswalk audited: {'MATCHED': 2239, 'UNMATCHED': 224, 'AMBIGUOUS': 44, 'DUPLICATE_OFFICIAL_ID': 17, 'LABEL_CONFLICT': 10}
Excluded unresolved/conflicting image identities: 295 of 2534
G02-A accepted cohort: 2239 {'BCN20000': 1240, 'HAM10000': 999} test coverage: 88.36%
G02-A per-class: label     AK  BCC  BKL  DF  MEL   NV  SCC  VASC
source                                         
BCN20000  74  275  111  14  288  420   48    10
HAM10000  11   56  113  10  110  669   15    15


## 3. Load exact original thesis Keras architecture and checkpoints

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from collections import OrderedDict
from IPython.display import display
try:
    for gpu in tf.config.list_physical_devices('GPU'):tf.config.experimental.set_memory_growth(gpu,True)
except RuntimeError:pass
print('TensorFlow',tf.__version__,'GPU:',tf.config.list_physical_devices('GPU'))
# These are the exact original Keras model builders from confidence-gated-ensemble, not PyTorch replacements.

# The original archive contains saved TensorFlow .weights.h5 models, not PyTorch checkpoints.
def locate_legacy(root):
    matches=sorted(p.parent.parent for p in root.rglob('rotation_ssl_encoder_best.weights.h5') if p.parent.name=='rotation_ssl')
    return sorted(set(matches))
roots=locate_legacy(INPUT)
if len(roots)!=1:
    # Saved Kaggle notebook _output_.zip can be nested. Never extract the 9.1 GB image ZIP here.
    STAGE=WORK/'.g02a_unpacked';STAGE.mkdir(exist_ok=True)
    for zp in sorted(INPUT.rglob('*.zip')):
        if zp.name=='ISIC_2019_Training_Input.zip':continue
        with zipfile.ZipFile(zp) as z:
            names=z.namelist();relevant=zp.name=='_output_.zip' or any(x.endswith('rotation_ssl/rotation_ssl_encoder_best.weights.h5') for x in names)
            if not relevant:continue
            size=sum(v.file_size for v in z.infolist());req(size<8*2**30,'Unexpected oversized original model archive')
            dest=STAGE/hashlib.sha256(str(zp).encode()).hexdigest()[:12];dest.mkdir(exist_ok=True)
            for v in z.infolist():
                if v.is_dir():continue
                rel=Path(v.filename.replace(chr(92),'/'));req(not rel.is_absolute() and '..' not in rel.parts,'Unsafe model archive')
                target=(dest/rel).resolve();req(target.is_relative_to(dest.resolve()),'Archive traversal')
                target.parent.mkdir(parents=True,exist_ok=True)
                if target.is_file() and target.stat().st_size==v.file_size:continue
                with z.open(v) as r,open(target,'wb') as w:shutil.copyfileobj(r,w,1<<20)
    roots=locate_legacy(INPUT)+locate_legacy(STAGE)
roots=sorted(set(roots));req(len(roots)==1,f'Attach one completed thesis source output with all model weights; found {roots}')
PIPELINE_ROOT=roots[0];print('Original models:',PIPELINE_ROOT)

TensorFlow 2.21.0 GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Original models: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike


In [4]:
# ============================================================
# EXTERNAL CELL E04: INFERENCE-ONLY MODEL ARCHITECTURES
# ============================================================

SIMCLR_DROPOUT = 0.30
ROTATION_FT_DROPOUT = 0.30

FUSIONCONCAT_DROPOUT = 0.40
FUSIONCONCAT_HIDDEN_DIM_1 = 512
FUSIONCONCAT_HIDDEN_DIM_2 = 256
FUSIONCONCAT_WEIGHT_DECAY = 1e-4

FUSIONGATED_V2_DROPOUT = 0.50
FUSIONGATED_V2_WEIGHT_DECAY = 3e-4
FUSIONGATED_V2_BRANCH_DIM = 256
FUSIONGATED_V2_GATE_HIDDEN = 128
FUSIONGATED_V2_HEAD_HIDDEN = 128
FUSIONGATED_V2_TEMPERATURE = 1.5

BESTV3_BRANCH_DIM = 256
BESTV3_DROPOUT_1 = 0.50
BESTV3_DROPOUT_2 = 0.40
BESTV3_WEIGHT_DECAY = 4e-4
BESTV3_GATE_HIDDEN = 128
BESTV3_HEAD_HIDDEN_1 = 256
BESTV3_HEAD_HIDDEN_2 = 128
BESTV3_TEMPERATURE = 1.0

RESIDUAL_FUSION_BRANCH_DIM = 256
RESIDUAL_FUSION_HIDDEN_DIM = 256
RESIDUAL_FUSION_WEIGHT_DECAY = 3e-4
RESIDUAL_FUSION_DROPOUT = 0.40
RESIDUAL_FUSION_MAX_ALPHA = 0.35


def resnet_preprocess_tensor(images):
    images = tf.cast(
        images,
        tf.float32,
    ) * 255.0

    return tf.keras.applications.resnet50.preprocess_input(
        images
    )


def build_resnet50_encoder(
    name,
    training_flag=False,
):
    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name=f"{name}_input",
    )

    backbone = ResNet50(
        include_top=False,
        weights=None,
        input_shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        pooling="avg",
    )

    features = backbone(
        inputs,
        training=training_flag,
    )

    return keras.Model(
        inputs,
        features,
        name=name,
    )


def build_simclr_finetune_model():
    encoder = build_resnet50_encoder(
        "simclr_resnet50_encoder",
        training_flag=False,
    )

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name="finetune_img",
    )

    x = layers.Lambda(
        resnet_preprocess_tensor,
        name="resnet_preprocess",
    )(inputs)

    x = encoder(
        x,
        training=False,
    )

    x = layers.Dropout(
        SIMCLR_DROPOUT,
        name="finetune_dropout",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="finetune_classifier",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="simclr_finetune_model",
    )

    return model, encoder


def build_rotation_finetune_model():
    encoder = build_resnet50_encoder(
        "rotation_ft_resnet50_encoder",
        training_flag=True,
    )
    encoder.trainable = True

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name="imagenet_rotation_input",
    )

    x = layers.Lambda(
        resnet_preprocess_tensor,
        name="resnet50_preprocess",
    )(inputs)

    # This reproduces the saved architecture.
    x = encoder(
        x,
        training=True
        if encoder.trainable
        else False,
    )

    x = layers.Dropout(
        ROTATION_FT_DROPOUT,
        name="imagenet_rotation_dropout",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="imagenet_rotation_classifier",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="imagenet_rotation_finetune_model",
    )

    return model


def l2_normalize_features(
    features,
    epsilon=1e-12,
):
    features = np.asarray(
        features,
        dtype=np.float32,
    )

    denominator = np.linalg.norm(
        features,
        axis=1,
        keepdims=True,
    )

    return (
        features
        / np.clip(
            denominator,
            epsilon,
            None,
        )
    ).astype(np.float32)


def build_fusionconcat_model(
    input_dim,
):
    inputs = keras.Input(
        shape=(input_dim,),
        name="fusionconcat_input",
    )

    x = layers.BatchNormalization(
        name="bn_input"
    )(inputs)

    x = layers.Dense(
        FUSIONCONCAT_HIDDEN_DIM_1,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONCONCAT_WEIGHT_DECAY
            )
        ),
        name="fusionconcat_dense_1",
    )(x)

    x = layers.BatchNormalization(
        name="bn_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_1",
    )(x)
    x = layers.Dropout(
        FUSIONCONCAT_DROPOUT,
        name="dropout_1",
    )(x)

    x = layers.Dense(
        FUSIONCONCAT_HIDDEN_DIM_2,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONCONCAT_WEIGHT_DECAY
            )
        ),
        name="fusionconcat_dense_2",
    )(x)

    x = layers.BatchNormalization(
        name="bn_2"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_2",
    )(x)
    x = layers.Dropout(
        FUSIONCONCAT_DROPOUT,
        name="dropout_2",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="fusionconcat_classifier",
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name="fusionconcat_model",
    )


def stable_softmax_temperature(
    values,
    temperature,
):
    return tf.nn.softmax(
        values / temperature,
        axis=-1,
    )


def build_fusiongated_v2_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_input",
    )

    simclr_norm = layers.LayerNormalization(
        name="ln_simclr_in"
    )(simclr_input)

    rotation_norm = layers.LayerNormalization(
        name="ln_rotation_in"
    )(rotation_input)

    simclr_projection = layers.Dense(
        FUSIONGATED_V2_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="simclr_proj",
    )(simclr_norm)

    simclr_projection = layers.LayerNormalization(
        name="ln_simclr_proj"
    )(simclr_projection)
    simclr_projection = layers.Activation(
        "relu",
        name="relu_simclr_proj",
    )(simclr_projection)

    rotation_projection = layers.Dense(
        FUSIONGATED_V2_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="rotation_proj",
    )(rotation_norm)

    rotation_projection = layers.LayerNormalization(
        name="ln_rotation_proj"
    )(rotation_projection)
    rotation_projection = layers.Activation(
        "relu",
        name="relu_rotation_proj",
    )(rotation_projection)

    gate_input = layers.Concatenate(
        name="gate_input_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    gate_hidden = layers.Dense(
        FUSIONGATED_V2_GATE_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="gate_dense_1",
    )(gate_input)

    gate_hidden = layers.LayerNormalization(
        name="ln_gate_1"
    )(gate_hidden)
    gate_hidden = layers.Activation(
        "relu",
        name="relu_gate_1",
    )(gate_hidden)
    gate_hidden = layers.Dropout(
        FUSIONGATED_V2_DROPOUT,
        name="dropout_gate_1",
    )(gate_hidden)

    gate_logits = layers.Dense(
        2,
        activation=None,
        bias_initializer=(
            keras.initializers.Constant(
                [0.30, 0.00]
            )
        ),
        name="gate_logits",
    )(gate_hidden)

    gate_weights = layers.Lambda(
        lambda values: (
            stable_softmax_temperature(
                values,
                FUSIONGATED_V2_TEMPERATURE,
            )
        ),
        name="branch_gate_softmax",
    )(gate_logits)

    alpha = layers.Lambda(
        lambda values: values[:, 0:1],
        name="alpha_weight",
    )(gate_weights)

    beta = layers.Lambda(
        lambda values: values[:, 1:2],
        name="beta_weight",
    )(gate_weights)

    weighted_simclr = layers.Multiply(
        name="weighted_simclr"
    )(
        [
            simclr_projection,
            alpha,
        ]
    )

    weighted_rotation = layers.Multiply(
        name="weighted_rotation"
    )(
        [
            rotation_projection,
            beta,
        ]
    )

    weighted_sum = layers.Add(
        name="weighted_sum"
    )(
        [
            weighted_simclr,
            weighted_rotation,
        ]
    )

    fused = layers.Concatenate(
        name="residual_fused_concat"
    )(
        [
            weighted_sum,
            simclr_projection,
            rotation_projection,
        ]
    )

    x = layers.LayerNormalization(
        name="ln_fused"
    )(fused)

    x = layers.Dense(
        FUSIONGATED_V2_HEAD_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="head_dense_1",
    )(x)

    x = layers.LayerNormalization(
        name="ln_head_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_1",
    )(x)
    x = layers.Dropout(
        FUSIONGATED_V2_DROPOUT,
        name="dropout_head_1",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="classifier",
    )(x)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
        ],
        outputs,
        name="fusiongated_v2_model",
    )


def build_fusiongated_v3_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_input",
    )

    simclr_norm = layers.LayerNormalization(
        name="ln_simclr_in"
    )(simclr_input)
    rotation_norm = layers.LayerNormalization(
        name="ln_rotation_in"
    )(rotation_input)

    simclr_projection = layers.Dense(
        BESTV3_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="simclr_proj",
    )(simclr_norm)
    simclr_projection = layers.LayerNormalization(
        name="ln_simclr_proj"
    )(simclr_projection)
    simclr_projection = layers.Activation(
        "relu",
        name="relu_simclr_proj",
    )(simclr_projection)

    rotation_projection = layers.Dense(
        BESTV3_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="rotation_proj",
    )(rotation_norm)
    rotation_projection = layers.LayerNormalization(
        name="ln_rotation_proj"
    )(rotation_projection)
    rotation_projection = layers.Activation(
        "relu",
        name="relu_rotation_proj",
    )(rotation_projection)

    gate_input = layers.Concatenate(
        name="gate_input_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    gate_hidden = layers.Dense(
        BESTV3_GATE_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="gate_dense_1",
    )(gate_input)
    gate_hidden = layers.LayerNormalization(
        name="ln_gate_1"
    )(gate_hidden)
    gate_hidden = layers.Activation(
        "relu",
        name="relu_gate_1",
    )(gate_hidden)
    gate_hidden = layers.Dropout(
        BESTV3_DROPOUT_1,
        name="dropout_gate_1",
    )(gate_hidden)

    gate_logits = layers.Dense(
        2,
        activation=None,
        bias_initializer=(
            keras.initializers.Constant(
                [0.35, 0.00]
            )
        ),
        name="gate_logits",
    )(gate_hidden)

    gate_weights = layers.Lambda(
        lambda values: (
            stable_softmax_temperature(
                values,
                BESTV3_TEMPERATURE,
            )
        ),
        name="branch_gate_softmax",
    )(gate_logits)

    alpha = layers.Lambda(
        lambda values: values[:, 0:1],
        name="alpha_weight",
    )(gate_weights)
    beta = layers.Lambda(
        lambda values: values[:, 1:2],
        name="beta_weight",
    )(gate_weights)

    weighted_simclr = layers.Multiply(
        name="weighted_simclr"
    )(
        [
            simclr_projection,
            alpha,
        ]
    )
    weighted_rotation = layers.Multiply(
        name="weighted_rotation"
    )(
        [
            rotation_projection,
            beta,
        ]
    )
    weighted_sum = layers.Add(
        name="weighted_sum"
    )(
        [
            weighted_simclr,
            weighted_rotation,
        ]
    )

    absolute_difference = layers.Lambda(
        lambda values: tf.abs(
            values[0] - values[1]
        ),
        name="abs_diff",
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    elementwise_product = layers.Multiply(
        name="elem_prod"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    fused = layers.Concatenate(
        name="final_fused_concat"
    )(
        [
            weighted_sum,
            simclr_projection,
            rotation_projection,
            absolute_difference,
            elementwise_product,
        ]
    )

    x = layers.LayerNormalization(
        name="ln_fused"
    )(fused)

    x = layers.Dense(
        BESTV3_HEAD_HIDDEN_1,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="head_dense_1",
    )(x)
    x = layers.LayerNormalization(
        name="ln_head_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_1",
    )(x)
    x = layers.Dropout(
        BESTV3_DROPOUT_1,
        name="dropout_head_1",
    )(x)

    x = layers.Dense(
        BESTV3_HEAD_HIDDEN_2,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="head_dense_2",
    )(x)
    x = layers.LayerNormalization(
        name="ln_head_2"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_2",
    )(x)
    x = layers.Dropout(
        BESTV3_DROPOUT_2,
        name="dropout_head_2",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="classifier",
    )(x)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
        ],
        outputs,
        name="fusiongated_best_v3_model",
    )


class BoundedResidualLogitFusion(
    layers.Layer
):
    def __init__(
        self,
        max_alpha=0.35,
        initial_alpha=0.08,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.max_alpha = float(
            max_alpha
        )
        ratio = np.clip(
            initial_alpha / max_alpha,
            1e-4,
            1.0 - 1e-4,
        )
        self.initial_logit = float(
            np.log(
                ratio
                / (1.0 - ratio)
            )
        )

    def build(
        self,
        input_shape,
    ):
        self.alpha_logit = (
            self.add_weight(
                name="alpha_logit",
                shape=(),
                initializer=(
                    keras.initializers.Constant(
                        self.initial_logit
                    )
                ),
                trainable=True,
            )
        )
        super().build(
            input_shape
        )

    def call(
        self,
        inputs,
    ):
        base_logits, correction_logits = inputs
        alpha = (
            self.max_alpha
            * tf.sigmoid(
                self.alpha_logit
            )
        )
        return (
            base_logits
            + alpha
            * correction_logits
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_alpha": self.max_alpha,
            }
        )
        return config


def build_residual_fusion_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_feature_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_feature_input",
    )
    base_logits_input = keras.Input(
        shape=(NUM_CLASSES,),
        name="base_simclr_logits",
    )

    simclr_projection = (
        layers.LayerNormalization(
            name="sim_ln_in"
        )(simclr_input)
    )
    simclr_projection = layers.Dense(
        RESIDUAL_FUSION_BRANCH_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="sim_projection",
    )(simclr_projection)
    simclr_projection = (
        layers.LayerNormalization(
            name="sim_ln_proj"
        )(simclr_projection)
    )
    simclr_projection = layers.Activation(
        "relu",
        name="sim_relu",
    )(simclr_projection)

    rotation_projection = (
        layers.LayerNormalization(
            name="rot_ln_in"
        )(rotation_input)
    )
    rotation_projection = layers.Dense(
        RESIDUAL_FUSION_BRANCH_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="rot_projection",
    )(rotation_projection)
    rotation_projection = (
        layers.LayerNormalization(
            name="rot_ln_proj"
        )(rotation_projection)
    )
    rotation_projection = layers.Activation(
        "relu",
        name="rot_relu",
    )(rotation_projection)

    absolute_difference = layers.Lambda(
        lambda values: tf.abs(
            values[0] - values[1]
        ),
        name="abs_difference",
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    product = layers.Multiply(
        name="elementwise_product"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    fused = layers.Concatenate(
        name="residual_feature_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
            absolute_difference,
            product,
        ]
    )
    fused = layers.LayerNormalization(
        name="fusion_ln"
    )(fused)
    fused = layers.Dense(
        RESIDUAL_FUSION_HIDDEN_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="correction_dense",
    )(fused)
    fused = layers.LayerNormalization(
        name="correction_ln"
    )(fused)
    fused = layers.Activation(
        "relu",
        name="correction_relu",
    )(fused)
    fused = layers.Dropout(
        RESIDUAL_FUSION_DROPOUT,
        name="correction_dropout",
    )(fused)
    correction_logits = layers.Dense(
        NUM_CLASSES,
        name="correction_logits",
    )(fused)

    final_logits = (
        BoundedResidualLogitFusion(
            max_alpha=(
                RESIDUAL_FUSION_MAX_ALPHA
            ),
            initial_alpha=0.08,
            name=(
                "bounded_residual_"
                "logit_fusion"
            ),
        )(
            [
                base_logits_input,
                correction_logits,
            ]
        )
    )

    output = layers.Softmax(
        dtype="float32",
        name="final_probabilities",
    )(final_logits)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
            base_logits_input,
        ],
        output,
        name=(
            "simclr_preserving_"
            "residual_fusion"
        ),
    )

In [5]:
# ============================================================
# EXTERNAL CELL E05: DISCOVER AND VERIFY SAVED MODEL WEIGHTS
# ============================================================

SIMCLR_DIR = (
    PIPELINE_ROOT
    / "simclr_finetune_multiseed_bacc_v2"
)

SIMCLR_CONFIG_JSON = (
    SIMCLR_DIR
    / "multiseed_config.json"
)

SIMCLR_RESULTS_CSV = (
    SIMCLR_DIR
    / "multiseed_results.csv"
)

SIMCLR_TTA_CONFIG_JSON = (
    SIMCLR_DIR
    / "tta_selection"
    / "selected_tta_policy.json"
)

with open(
    SIMCLR_CONFIG_JSON,
    "r",
) as file:
    simclr_config = json.load(file)

SIMCLR_SEEDS = [
    int(seed)
    for seed in simclr_config[
        "seeds"
    ]
]

SIMCLR_SEED_PATHS = OrderedDict(
    [
        (
            seed,
            SIMCLR_DIR
            / f"seed_{seed}"
            / "best_by_val_bacc.weights.h5",
        )
        for seed in SIMCLR_SEEDS
    ]
)

simclr_results_df = pd.read_csv(
    SIMCLR_RESULTS_CSV
)

representative_row = (
    simclr_results_df.sort_values(
        [
            "val_bacc",
            "val_macro_f1",
            "val_accuracy",
            "seed",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    ).iloc[0]
)

SIMCLR_REPRESENTATIVE_SEED = int(
    representative_row["seed"]
)

with open(
    SIMCLR_TTA_CONFIG_JSON,
    "r",
) as file:
    selected_tta_config = json.load(
        file
    )

SELECTED_TTA_TRANSFORMS = list(
    selected_tta_config[
        "transforms"
    ]
)

ROTATION_SSL_ENCODER_PATH = (
    PIPELINE_ROOT
    / "rotation_ssl"
    / "rotation_ssl_encoder_best.weights.h5"
)

ROTATION_FT_PATH = (
    PIPELINE_ROOT
    / "extended_imagenet_rotation_finetune"
    / "imagenet_rotation_finetune_best.weights.h5"
)

FUSIONCONCAT_PATH = (
    PIPELINE_ROOT
    / "fusion_concat_multiseed_rep_v2"
    / "fusionconcat_best.weights.h5"
)

FUSIONGATED_V2_PATH = (
    PIPELINE_ROOT
    / "fusion_gated_v2_multiseed_rep_v2"
    / "fusiongated_v2_best.weights.h5"
)

FUSIONGATED_V3_PATH = (
    PIPELINE_ROOT
    / "fusion_gated_v3_multiseed_rep_v2"
    / "fusiongated_best_v3.weights.h5"
)

RESIDUAL_DIR = (
    PIPELINE_ROOT
    / "residual_simclr_rotation_fusion_v1"
)

RESIDUAL_CONFIG_JSON = (
    RESIDUAL_DIR
    / "config.json"
)

with open(
    RESIDUAL_CONFIG_JSON,
    "r",
) as file:
    residual_config = json.load(
        file
    )

RESIDUAL_SEEDS = [
    int(seed)
    for seed in residual_config[
        "seeds"
    ]
]

RESIDUAL_SEED_PATHS = OrderedDict(
    [
        (
            seed,
            RESIDUAL_DIR
            / f"seed_{seed}"
            / "best_by_val_bacc.weights.h5",
        )
        for seed in RESIDUAL_SEEDS
    ]
)

artifact_rows = []

for name, path in {
    **{
        f"SimCLR seed {seed}": path
        for seed, path
        in SIMCLR_SEED_PATHS.items()
    },
    "Rotation SSL encoder": (
        ROTATION_SSL_ENCODER_PATH
    ),
    "Standalone Rotation": (
        ROTATION_FT_PATH
    ),
    "FusionConcat": (
        FUSIONCONCAT_PATH
    ),
    "FusionGated V2": (
        FUSIONGATED_V2_PATH
    ),
    "FusionGated V3": (
        FUSIONGATED_V3_PATH
    ),
    **{
        f"Residual Fusion seed {seed}": path
        for seed, path
        in RESIDUAL_SEED_PATHS.items()
    },
}.items():
    artifact_rows.append(
        {
            "artifact": name,
            "path": str(path),
            "exists": path.exists(),
        }
    )

ARTIFACT_DF = pd.DataFrame(
    artifact_rows
)

display(ARTIFACT_DF)

missing_artifacts = ARTIFACT_DF.loc[
    ~ARTIFACT_DF["exists"]
]

if len(missing_artifacts) > 0:
    raise FileNotFoundError(
        "Some saved inference artifacts are missing:\n"
        + missing_artifacts.to_string(
            index=False
        )
    )

print(
    "Representative SimCLR seed:",
    SIMCLR_REPRESENTATIVE_SEED,
)
print(
    "Selected TTA transforms:",
    SELECTED_TTA_TRANSFORMS,
)
print(
    "Residual seeds:",
    RESIDUAL_SEEDS,
)

                   artifact  \
0            SimCLR seed 21   
1            SimCLR seed 35   
2            SimCLR seed 42   
3            SimCLR seed 52   
4            SimCLR seed 62   
5      Rotation SSL encoder   
6       Standalone Rotation   
7              FusionConcat   
8            FusionGated V2   
9            FusionGated V3   
10  Residual Fusion seed 21   
11  Residual Fusion seed 42   
12  Residual Fusion seed 62   

                                                 path  exists  
0   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
1   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
2   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
3   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
4   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
5   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
6   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
7   /kaggle/input/notebooks/anirbanroysudipto/last...

Representative SimCLR seed: 62
Selected TTA transforms: ['identity', 'hflip']
Residual seeds: [21, 42, 62]


## 4. Original thesis inference functions and 224px data loader

In [6]:
# ============================================================
# EXTERNAL CELL E06: BASE-MODEL EXTERNAL INFERENCE WITH CACHING
# ============================================================

def apply_tta_transform(
    images,
    transform_name,
):
    if transform_name == "identity":
        return images
    if transform_name == "hflip":
        return tf.image.flip_left_right(
            images
        )
    if transform_name == "vflip":
        return tf.image.flip_up_down(
            images
        )
    if transform_name == "rot90":
        return tf.image.rot90(
            images,
            k=1,
        )
    if transform_name == "rot180":
        return tf.image.rot90(
            images,
            k=2,
        )
    if transform_name == "rot270":
        return tf.image.rot90(
            images,
            k=3,
        )

    raise ValueError(
        f"Unknown TTA transform: {transform_name}"
    )


def predict_simclr_seed_both(
    model,
    dataset,
    tta_transforms,
):
    no_tta_batches = []
    selected_tta_batches = []
    labels = []

    for images, batch_labels in dataset:
        identity_probability = np.asarray(
            model(
                images,
                training=False,
            ),
            dtype=np.float32,
        )

        view_probabilities = []

        for transform_name in tta_transforms:
            if transform_name == "identity":
                transformed_probability = (
                    identity_probability
                )
            else:
                transformed_images = (
                    apply_tta_transform(
                        images,
                        transform_name,
                    )
                )
                transformed_probability = (
                    np.asarray(
                        model(
                            transformed_images,
                            training=False,
                        ),
                        dtype=np.float32,
                    )
                )

            view_probabilities.append(
                transformed_probability
            )

        selected_probability = np.mean(
            np.stack(
                view_probabilities,
                axis=0,
            ),
            axis=0,
        )

        no_tta_batches.append(
            identity_probability
        )
        selected_tta_batches.append(
            selected_probability
        )
        labels.append(
            np.asarray(
                batch_labels,
                dtype=np.int32,
            )
        )

    return (
        normalize_probabilities(
            np.concatenate(
                no_tta_batches,
                axis=0,
            )
        ),
        normalize_probabilities(
            np.concatenate(
                selected_tta_batches,
                axis=0,
            )
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
    )


def extract_encoder_features(
    encoder,
    dataset,
):
    feature_batches = []
    labels = []

    for images, batch_labels in dataset:
        preprocessed = (
            resnet_preprocess_tensor(
                images
            )
        )

        features = encoder(
            preprocessed,
            training=False,
        )

        feature_batches.append(
            np.asarray(
                features,
                dtype=np.float32,
            )
        )

        labels.append(
            np.asarray(
                batch_labels,
                dtype=np.int32,
            )
        )

    return (
        l2_normalize_features(
            np.concatenate(
                feature_batches,
                axis=0,
            )
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
    )


def predict_direct_model(
    model,
    dataset,
):
    probability_batches = []
    labels = []

    for images, batch_labels in dataset:
        probability_batches.append(
            np.asarray(
                model(
                    images,
                    training=False,
                ),
                dtype=np.float32,
            )
        )
        labels.append(
            np.asarray(
                batch_labels,
                dtype=np.int32,
            )
        )

    return (
        normalize_probabilities(
            np.concatenate(
                probability_batches,
                axis=0,
            )
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
    )


def generate_external_base_probabilities(
    dataset_name,
    dataset,
    expected_labels,
):
    cache_path = (
        EXTERNAL_CACHE_DIR
        / f"{dataset_name}_all_base_probabilities.npz"
    )

    if cache_path.exists():
        with np.load(
            cache_path
        ) as saved:
            cached_names = [
                str(value)
                for value in saved[
                    "model_names"
                ].tolist()
            ]

            if cached_names == MODEL_NAMES:
                cached_labels = np.asarray(
                    saved["labels"],
                    dtype=np.int32,
                )

                if np.array_equal(
                    cached_labels,
                    expected_labels,
                ):
                    print(
                        "Using cached external base probabilities:",
                        cache_path,
                    )

                    return OrderedDict(
                        [
                            (
                                model_name,
                                normalize_probabilities(
                                    saved[
                                        f"prob__{model_name}"
                                    ]
                                ),
                            )
                            for model_name
                            in MODEL_NAMES
                        ]
                    )

    print(
        "Generating external base probabilities for",
        dataset_name,
    )

    # --------------------------------------------------------
    # 1. Multi-seed SimCLR: no-TTA and selected TTA.
    # --------------------------------------------------------
    no_tta_seed_probabilities = []
    selected_tta_seed_probabilities = []
    representative_probability = None

    for seed, weight_path in (
        SIMCLR_SEED_PATHS.items()
    ):
        print(
            f"SimCLR external inference, seed {seed}"
        )

        tf.keras.backend.clear_session()
        gc.collect()

        model, _ = (
            build_simclr_finetune_model()
        )
        model.load_weights(
            str(weight_path)
        )

        (
            seed_no_tta,
            seed_selected_tta,
            labels_check,
        ) = predict_simclr_seed_both(
            model,
            dataset,
            SELECTED_TTA_TRANSFORMS,
        )

        if not np.array_equal(
            labels_check,
            expected_labels,
        ):
            raise ValueError(
                "External label order changed during "
                f"SimCLR seed {seed} inference."
            )

        no_tta_seed_probabilities.append(
            seed_no_tta
        )
        selected_tta_seed_probabilities.append(
            seed_selected_tta
        )

        if (
            seed
            == SIMCLR_REPRESENTATIVE_SEED
        ):
            representative_probability = (
                seed_no_tta
            )

        del model
        gc.collect()

    simclr_no_tta_probability = (
        normalize_probabilities(
            np.mean(
                np.stack(
                    no_tta_seed_probabilities,
                    axis=0,
                ),
                axis=0,
            )
        )
    )

    simclr_selected_tta_probability = (
        normalize_probabilities(
            np.mean(
                np.stack(
                    selected_tta_seed_probabilities,
                    axis=0,
                ),
                axis=0,
            )
        )
    )

    if representative_probability is None:
        raise RuntimeError(
            "The representative SimCLR seed probability "
            "was not generated."
        )

    # --------------------------------------------------------
    # 2. Extract representative SimCLR and Rotation features.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    representative_model, representative_encoder = (
        build_simclr_finetune_model()
    )
    representative_model.load_weights(
        str(
            SIMCLR_SEED_PATHS[
                SIMCLR_REPRESENTATIVE_SEED
            ]
        )
    )

    (
        simclr_features,
        feature_labels,
    ) = extract_encoder_features(
        representative_encoder,
        dataset,
    )

    del representative_model
    del representative_encoder
    gc.collect()

    if not np.array_equal(
        feature_labels,
        expected_labels,
    ):
        raise ValueError(
            "SimCLR external feature label order changed."
        )

    tf.keras.backend.clear_session()
    gc.collect()

    rotation_encoder = build_resnet50_encoder(
        "rotation_resnet50_encoder",
        training_flag=False,
    )
    rotation_encoder.load_weights(
        str(
            ROTATION_SSL_ENCODER_PATH
        )
    )
    rotation_encoder.trainable = False

    (
        rotation_features,
        rotation_feature_labels,
    ) = extract_encoder_features(
        rotation_encoder,
        dataset,
    )

    del rotation_encoder
    gc.collect()

    if not np.array_equal(
        rotation_feature_labels,
        expected_labels,
    ):
        raise ValueError(
            "Rotation external feature label order changed."
        )

    concat_features = np.concatenate(
        [
            simclr_features,
            rotation_features,
        ],
        axis=1,
    ).astype(np.float32)

    # --------------------------------------------------------
    # 3. Standalone Rotation.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    rotation_model = (
        build_rotation_finetune_model()
    )
    rotation_model.load_weights(
        str(ROTATION_FT_PATH)
    )

    (
        rotation_probability,
        rotation_labels,
    ) = predict_direct_model(
        rotation_model,
        dataset,
    )

    del rotation_model
    gc.collect()

    if not np.array_equal(
        rotation_labels,
        expected_labels,
    ):
        raise ValueError(
            "Standalone Rotation external label order changed."
        )

    # --------------------------------------------------------
    # 4. FusionConcat.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    fusionconcat_model = (
        build_fusionconcat_model(
            concat_features.shape[1]
        )
    )
    fusionconcat_model.load_weights(
        str(FUSIONCONCAT_PATH)
    )

    fusionconcat_probability = (
        normalize_probabilities(
            fusionconcat_model.predict(
                concat_features,
                batch_size=256,
                verbose=0,
            )
        )
    )

    del fusionconcat_model
    gc.collect()

    # --------------------------------------------------------
    # 5. FusionGated V2.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    fusiongated_v2_model = (
        build_fusiongated_v2_model(
            simclr_features.shape[1],
            rotation_features.shape[1],
        )
    )
    fusiongated_v2_model.load_weights(
        str(FUSIONGATED_V2_PATH)
    )

    fusiongated_v2_probability = (
        normalize_probabilities(
            fusiongated_v2_model.predict(
                [
                    simclr_features,
                    rotation_features,
                ],
                batch_size=256,
                verbose=0,
            )
        )
    )

    del fusiongated_v2_model
    gc.collect()

    # --------------------------------------------------------
    # 6. FusionGated V3.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    fusiongated_v3_model = (
        build_fusiongated_v3_model(
            simclr_features.shape[1],
            rotation_features.shape[1],
        )
    )
    fusiongated_v3_model.load_weights(
        str(FUSIONGATED_V3_PATH)
    )

    fusiongated_v3_probability = (
        normalize_probabilities(
            fusiongated_v3_model.predict(
                [
                    simclr_features,
                    rotation_features,
                ],
                batch_size=256,
                verbose=0,
            )
        )
    )

    del fusiongated_v3_model
    gc.collect()

    # --------------------------------------------------------
    # 7. Residual Fusion multi-seed.
    # --------------------------------------------------------
    representative_logits = np.log(
        np.clip(
            representative_probability,
            1e-7,
            1.0,
        )
    ).astype(np.float32)

    residual_probabilities = []

    for seed, weight_path in (
        RESIDUAL_SEED_PATHS.items()
    ):
        print(
            f"Residual Fusion external inference, seed {seed}"
        )

        tf.keras.backend.clear_session()
        gc.collect()

        residual_model = (
            build_residual_fusion_model(
                simclr_features.shape[1],
                rotation_features.shape[1],
            )
        )
        residual_model.load_weights(
            str(weight_path)
        )

        residual_probabilities.append(
            normalize_probabilities(
                residual_model.predict(
                    [
                        simclr_features,
                        rotation_features,
                        representative_logits,
                    ],
                    batch_size=256,
                    verbose=0,
                )
            )
        )

        del residual_model
        gc.collect()

    residual_probability = (
        normalize_probabilities(
            np.mean(
                np.stack(
                    residual_probabilities,
                    axis=0,
                ),
                axis=0,
            )
        )
    )

    source_probabilities = OrderedDict(
        [
            (
                "SimCLR_NoTTA",
                simclr_no_tta_probability,
            ),
            (
                "SimCLR_SelectedTTA",
                simclr_selected_tta_probability,
            ),
            (
                "Rotation",
                rotation_probability,
            ),
            (
                "FusionConcat",
                fusionconcat_probability,
            ),
            (
                "FusionGatedV2",
                fusiongated_v2_probability,
            ),
            (
                "FusionGatedV3",
                fusiongated_v3_probability,
            ),
            (
                "ResidualFusion",
                residual_probability,
            ),
        ]
    )

    if list(
        source_probabilities.keys()
    ) != MODEL_NAMES:
        raise ValueError(
            "External source ordering does not match "
            "the frozen stacking model."
        )

    save_payload = {
        "model_names": np.asarray(
            MODEL_NAMES
        ),
        "labels": expected_labels,
    }

    for model_name, probability in (
        source_probabilities.items()
    ):
        save_payload[
            f"prob__{model_name}"
        ] = probability.astype(
            np.float32
        )

    np.savez_compressed(
        cache_path,
        **save_payload,
    )

    print(
        "Saved external base-probability cache:",
        cache_path,
    )

    return source_probabilities

In [7]:
def normalize_probabilities(p):
    p=np.asarray(p,dtype='float64')
    req(p.ndim==2 and p.shape[1]==8 and np.isfinite(p).all(),'Invalid probability matrix')
    p=np.clip(p,1e-12,None);return p/p.sum(1,keepdims=True)
def build_dataset(records):
    names=records.hf_id.astype(str).tolist();labels=records.label_id.astype('int32').tolist()
    def images():
        with zipfile.ZipFile(HF_ARCHIVE) as z:
            for name,y in zip(names,labels):
                blob=z.read(HF_MEMBERS[name]);req(len(blob)>100,'Corrupt original 224px image '+name)
                yield blob,np.int32(y)
    ds=tf.data.Dataset.from_generator(images,output_signature=(tf.TensorSpec((),tf.string),tf.TensorSpec((),tf.int32)))
    def parse(b,y):
        x=tf.io.decode_image(b,channels=3,expand_animations=False);x.set_shape((None,None,3))
        x=tf.image.resize(tf.cast(x,tf.float32),(224,224),method='bilinear')/255.
        return x,y
    return ds.map(parse,num_parallel_calls=2,deterministic=True).batch(EXTERNAL_BATCH_SIZE).prefetch(1)
x,y=next(iter(build_dataset(df.iloc[:2])))
req(tuple(x.shape[1:])==(224,224,3),'Wrong original thesis input dimensions')
print('Original thesis 224px input verified')

Original thesis 224px input verified


## 5. Frozen original model inference, chunking, and resumable output

In [8]:
# Only copy matching G02-A results, never G02-B chunks.
for pm in INPUT.rglob('g02a_status.json'):
    previous=json.loads(pm.read_text())
    if previous.get('g01_cohort_sha256')!=info['cohort_sha256'] or previous.get('j1_split_sha256')!=sha(j1):continue
    for p in (pm.parent.parent/'predictions').glob('base_*.npz'):
        target=OUT/'predictions'/p.name
        if not target.exists():shutil.copy2(p,target)
rows=[]
for start in range(0,len(df),CHUNK):
    end=min(start+CHUNK,len(df));part=df.iloc[start:end].reset_index(drop=True)
    token=hashlib.sha256('\n'.join(part.hf_id+'|'+part.image_id).encode()).hexdigest()[:16]
    path=OUT/'predictions'/f'base_{start:05d}_{end:05d}_{token}.npz'
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            req(np.array_equal(z['image_id'].astype(str),part.image_id.to_numpy()),'Cached official IDs changed')
            req(np.array_equal(z['hf_id'].astype(str),part.hf_id.to_numpy()),'Cached J1 IDs changed')
            req(np.array_equal(z['y'],part.label_id.to_numpy(dtype='int32')),'Cached labels changed')
            req(all('prob__'+s in z for s in MODEL_NAMES),'Incomplete cache')
        rows.append({'file':path.name,'sha256':sha(path),'start':start,'end':end});print('REUSE',end);continue
    if time.monotonic()-START>MAX_HOURS*3600:print('PAUSED at',start,'for Kaggle runtime');break
    out=generate_external_base_probabilities('g02a_'+token,build_dataset(part),part.label_id.to_numpy(dtype='int32'))
    req(list(out)==MODEL_NAMES,'Legacy source order changed')
    payload={'image_id':part.image_id.to_numpy(dtype=str),'hf_id':part.hf_id.to_numpy(dtype=str),
      'source':part.source.to_numpy(dtype=str),'group_id':part.source_group.to_numpy(dtype=str),
      'y':part.label_id.to_numpy(dtype='int32'),'classes':np.asarray(CLASSES),'model_names':np.asarray(MODEL_NAMES),
      'j1_split_sha256':np.asarray(sha(j1))}
    for s,p in out.items():
        a=np.asarray(p,dtype=np.float32)
        req(a.shape==(len(part),8) and np.isfinite(a).all(),'Malformed model probabilities')
        req(np.max(abs(a.sum(1)-1))<1e-4,'Non-normalized model predictions')
        payload['prob__'+s]=a
    np.savez_compressed(path,**payload);rows.append({'file':path.name,'sha256':sha(path),'start':start,'end':end})
    print('SAVED',end,'/',len(df));gc.collect()
processed=sum(r['end']-r['start'] for r in rows)
status={'stage':'G02-A','status':'COMPLETED' if processed==len(df) else 'PAUSED','images':len(df),'processed':processed,
  'classes':CLASSES,'models':MODEL_NAMES,'chunks':rows,'j1_split_sha256':sha(j1),'g01_cohort_sha256':info['cohort_sha256'],
  'scope':'Original thesis J1 held-out images matched by content to HAM/BCN within ISIC2019; NOT source-independent generalization'}
writejson(OUT/'reports/g02a_status.json',status)
print(status['status'],processed,'/',len(df))

Generating external base probabilities for g02a_9394935d3220b0ca
SimCLR external inference, seed 21
SimCLR external inference, seed 35
SimCLR external inference, seed 42
SimCLR external inference, seed 52
SimCLR external inference, seed 62
Residual Fusion external inference, seed 21
Residual Fusion external inference, seed 42
Residual Fusion external inference, seed 62
Saved external base-probability cache: /kaggle/working/g02a-image-level-ham-bcn-v2/cache/g02a_9394935d3220b0ca_all_base_probabilities.npz
SAVED 384 / 2239
Generating external base probabilities for g02a_c197e0e63dbbf8a4
SimCLR external inference, seed 21
SimCLR external inference, seed 35
SimCLR external inference, seed 42
SimCLR external inference, seed 52
SimCLR external inference, seed 62
Residual Fusion external inference, seed 21
Residual Fusion external inference, seed 42
Residual Fusion external inference, seed 62
Saved external base-probability cache: /kaggle/working/g02a-image-level-ham-bcn-v2/cache/g02a_c197e0e

## 6. Optional original five-seed supervised ResNet50 (only if its saved weights are attached)

In [9]:
# This source was trained in the separate supervised-classification-for-journal notebook.
# Only evaluate when its EXACT weight files are attached. Never substitute ImageNet-only weights.
sup_roots=sorted(set(p.parents[2] for p in INPUT.rglob('best_by_val_bacc.weights.h5') if 'mbec_plain_supervised_resnet50' in p.parts and p.parent.name.startswith('seed_')))
if len(sup_roots)>1:raise RuntimeError(f'Ambiguous supervised outputs: {sup_roots}')
if sup_roots:
    SUP=sup_roots[0];sup_seeds=[21,35,42,52,62]
    paths=[SUP/'models'/f'seed_{seed}'/'best_by_val_bacc.weights.h5' for seed in sup_seeds]
    req(all(p.is_file() for p in paths),'Supervised baseline output incomplete; missing at least one trained seed')
    def create_supervised(seed):
        inp=keras.Input((224,224,3),name='image')
        x=layers.Lambda(lambda v:tf.keras.applications.resnet50.preprocess_input(tf.cast(v,tf.float32)*255.0),name='resnet50_preprocess')(inp)
        backbone=ResNet50(include_top=False,weights=None,input_shape=(224,224,3),pooling='avg')
        x=backbone(x,training=False);x=layers.Dropout(.30,name='dropout')(x)
        out=layers.Dense(8,activation='softmax',dtype='float32',name='classifier')(x)
        return keras.Model(inp,out,name=f'supervised_resnet50_seed_{seed}')
    for start in range(0,len(df),CHUNK):
        end=min(start+CHUNK,len(df));part=df.iloc[start:end].reset_index(drop=True)
        token=hashlib.sha256(('|'.join(part.image_id)).encode()).hexdigest()[:16]
        dest=OUT/'predictions'/f'supervised_{start:05d}_{end:05d}_{token}.npz'
        if dest.exists():continue
        pred=[]
        for seed,p in zip(sup_seeds,paths):
            tf.keras.backend.clear_session();m=create_supervised(seed);m.load_weights(str(p))
            a=m.predict(build_dataset(part),verbose=0);req(a.shape==(len(part),8),'Supervised prediction shape mismatch');pred.append(a)
            del m;gc.collect()
        prob=normalize_probabilities(np.mean(np.stack(pred),axis=0))
        np.savez_compressed(dest,image_id=part.image_id.to_numpy(dtype=str),source=part.source.to_numpy(dtype=str),y=part.label_id.to_numpy(dtype='int32'),classes=np.array(CLASSES),proba=prob.astype('float32'),model_id=np.array('Supervised_ResNet50_5seed'))
        print('Saved supervised',end,'/',len(df))
    print('Original supervised baseline successfully evaluated')
else:print('No original five-seed supervised weights attached. G02-A seven core sources complete; supervised predictions NOT evaluated.')

No original five-seed supervised weights attached. G02-A seven core sources complete; supervised predictions NOT evaluated.


## 7. Eight-output source-specific reference metrics

In [10]:
if status['status']=='COMPLETED':
    probs={s:[] for s in MODEL_NAMES};ys=[];srcs=[];ids=[]
    for r in rows:
        with np.load(OUT/'predictions'/r['file'],allow_pickle=False) as z:
            ids.extend(z['image_id'].astype(str));ys.extend(z['y'].astype(int));srcs.extend(z['source'].astype(str))
            for s in MODEL_NAMES:probs[s].append(np.asarray(z['prob__'+s],dtype=np.float32))
    req(ids==df.image_id.tolist(),'Chunk output IDs differ from validated cohort')
    y=np.asarray(ys);source=np.asarray(srcs);results=[];recalls=[]
    for s in MODEL_NAMES:
        p=np.concatenate(probs[s]);pred=p.argmax(1)
        for k in ['HAM10000','BCN20000']:
            j=np.where(source==k)[0]
            if not len(j):continue
            yy=y[j];pp=pred[j];seen=np.unique(yy)
            results.append({'source':k,'model':s,'images':len(j),'balanced_accuracy':float(balanced_accuracy_score(yy,pp)),
                'accuracy':float(accuracy_score(yy,pp)),
                'macro_f1_present_classes':float(f1_score(yy,pp,labels=seen,average='macro',zero_division=0)),
                'unsupported_prediction_rate':float((~np.isin(pp,seen)).mean()),
                'present_classes':','.join(CLASSES[c] for c in seen)})
            for c in seen:
                mask=yy==c
                recalls.append({'source':k,'model':s,'class':CLASSES[c],'support':int(mask.sum()),'recall':float((pp[mask]==c).mean())})
    pd.DataFrame(results).to_csv(OUT/'tables/source_model_metrics.csv',index=False)
    pd.DataFrame(recalls).to_csv(OUT/'tables/source_class_recall.csv',index=False)
    print(pd.DataFrame(results)[['source','model','images','balanced_accuracy','unsupported_prediction_rate']].to_string(index=False))
else:print('Run remaining chunks before computing final metrics')
# Supervised scores are intentionally a separate artifact from the seven original fusion/SSL sources.
if status['status']=='COMPLETED':
    files=sorted((OUT/'predictions').glob('supervised_*.npz'))
    if files and len(files)==len(rows):
        by_ids={}
        for f in files:
            with np.load(f,allow_pickle=False) as z:
                for ident,probs in zip(z['image_id'].astype(str),z['proba'].astype(np.float32)):
                    req(ident not in by_ids,'Duplicate supervised probability image ID')
                    by_ids[ident]=probs
        req(set(by_ids)==set(df.image_id),'Supervised predictions do not cover the same G02-A cohort')
        p=np.stack([by_ids[x] for x in df.image_id]);pred=p.argmax(1)
        vals=[]
        for src in ['HAM10000','BCN20000']:
            subset=df.source.eq(src).to_numpy();yy=df.label_id.to_numpy()[subset];pp=pred[subset]
            present=np.unique(yy)
            vals.append({'source':src,'model':'Supervised_ResNet50_5seed','images':len(yy),
                'balanced_accuracy':float(balanced_accuracy_score(yy,pp)),
                'accuracy':float(accuracy_score(yy,pp)),
                'macro_f1_present_classes':float(f1_score(yy,pp,labels=present,average='macro',zero_division=0)),
                'unsupported_prediction_rate':float((~np.isin(pp,present)).mean()),
                'present_classes':','.join(CLASSES[int(c)] for c in present)})
        pd.DataFrame(vals).to_csv(OUT/'tables/source_supervised_optional.csv',index=False)
        print('Additional saved supervised source:',pd.DataFrame(vals).to_string(index=False))
    elif files:print('Supervised optional subset incomplete; not reporting scores')


  source              model  images  balanced_accuracy  unsupported_prediction_rate
HAM10000       SimCLR_NoTTA     999           0.763616                          0.0
BCN20000       SimCLR_NoTTA    1240           0.815408                          0.0
HAM10000 SimCLR_SelectedTTA     999           0.770121                          0.0
BCN20000 SimCLR_SelectedTTA    1240           0.806494                          0.0
HAM10000           Rotation     999           0.751362                          0.0
BCN20000           Rotation    1240           0.720479                          0.0
HAM10000       FusionConcat     999           0.779636                          0.0
BCN20000       FusionConcat    1240           0.769950                          0.0
HAM10000      FusionGatedV2     999           0.757311                          0.0
BCN20000      FusionGatedV2    1240           0.781515                          0.0
HAM10000      FusionGatedV3     999           0.737211                      